# SAE steering side effects: collateral-side residualization, EDA and figures

Companion notebook for the four-setting extension of *Pre-Intervention Prediction of Sparse Autoencoder Steering Side Effects* (Duan, [arXiv:2606.08365](https://arxiv.org/abs/2606.08365)). The paper controls its stability labels for effect magnitude, intervention value and natural activation but never applies that control to its collateral labels. The attached dataset holds the per-feature predictors and steering labels measured from scratch in all four of the paper's settings (GPT-2-small, Pythia-70M-deduped, Gemma-2-2B, Llama-3.1-8B), and this notebook walks through the data and the controlled results. Code and write-up: [github.com/yashb98/sae-steering-collateral-residualization](https://github.com/yashb98/sae-steering-collateral-residualization).

The last section re-runs the GPT-2-small and Pythia settings on this Kaggle GPU and compares the headline statistics with the dataset. Gemma-2-2B and Llama-3.1-8B are not recomputed here: both are gated on Hugging Face and the 8B model does not fit the Kaggle session; their rows come from the dataset.

In [ ]:
import glob, json, os, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
warnings.filterwarnings("ignore")

# ---- data location: the Kaggle dataset when attached, otherwise a local results/ directory ----
cands = glob.glob("/kaggle/input/*/results") + glob.glob("/kaggle/input/*") + ["results", "../results"]
DATA = next(p for p in cands if os.path.exists(os.path.join(p, "gpt2_small", "per_feature.csv")))
print("data:", DATA)

ORDER = ["gpt2_small", "pythia_70m_deduped", "gemma_2_2b", "llama_3_1_8b"]
PRETTY = {"gpt2_small": "GPT-2-small", "pythia_70m_deduped": "Pythia-70M", "gemma_2_2b": "Gemma-2-2B", "llama_3_1_8b": "Llama-3.1-8B"}
COLOR = {"gpt2_small": "#2a78d6", "pythia_70m_deduped": "#eb6834", "gemma_2_2b": "#1baf7a", "llama_3_1_8b": "#4a3aa7"}
CONTROL_COLOR = {"none": "#86b6ef", "robust": "#2a78d6", "primary": "#104281"}
CONTROL_LABEL = {"none": "no control", "robust": "frequency + activation magnitude", "primary": "Section 3.8 set + frequency"}
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
DIVERGING = LinearSegmentedColormap.from_list("bluegrayred", ["#2a78d6", "#f0efec", "#e34948"])
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK2, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titlecolor": INK, "legend.frameon": False, "font.family": "sans-serif"})

part = pd.read_csv(os.path.join(DATA, "analysis", "partial_correlations.csv"))
settings = [s for s in ORDER if os.path.exists(os.path.join(DATA, s, "per_feature.csv")) and s in set(part.setting)]
feat = {s: pd.read_csv(os.path.join(DATA, s, "per_feature.csv")) for s in settings}
meta = {s: json.load(open(os.path.join(DATA, s, "meta.json"))) for s in settings}
cvr = pd.read_csv(os.path.join(DATA, "analysis", "residualized_cv_ridge.csv"))
seeds = pd.read_csv(os.path.join(DATA, "analysis", "seed_summary.csv"))
xcheck_path = os.path.join(DATA, "crosscheck_kaggle_t4", "headline.json")
xcheck = json.load(open(xcheck_path)) if os.path.exists(xcheck_path) else None
gate_path = os.path.join(DATA, "gpt2_small_v1", "meta.json")
gate = json.load(open(gate_path)) if os.path.exists(gate_path) else None

inv = pd.DataFrame([{"setting": PRETTY[s], "features": len(feat[s]), "eligible": meta[s]["sizes"]["n_eligible"],
                     "d_sae": meta[s]["sizes"]["d_sae_primary"], "mean L0 (primary)": meta[s]["sizes"].get("mean_l0_primary"),
                     "panel": meta[s]["sizes"]["panel"], "dtype": meta[s]["dtype"], "wall clock (s)": meta[s]["wall_clock_s"],
                     "device": meta[s]["device"]} for s in settings]).set_index("setting")
inv

## 1. What the sampled features look like

Each setting has 300 features sampled with seed 0 from the final-token firing-frequency band [0.002, 0.50]. The predictors are computed before any steering; the labels come from steering each feature additively (alpha = 1.0) at the final token of 48 mixed contexts and measuring what moves downstream.

In [ ]:
KEY = ["crowding", "frequency", "act_mag", "effect_l2", "collateral_raw", "collateral_ctilde"]
LABEL = {"crowding": "decoder crowding (top-20 mean |cos|)", "frequency": "firing frequency", "act_mag": "mean activation",
         "effect_l2": "effect magnitude E_f", "collateral_raw": "collateral count C", "collateral_ctilde": "C-tilde = C / E_f"}
rows = []
for s in settings:
    q = feat[s][KEY].quantile([0.1, 0.5, 0.9]).T
    for k in KEY:
        rows.append({"setting": PRETTY[s], "variable": LABEL[k], "p10": q.loc[k, 0.1], "median": q.loc[k, 0.5], "p90": q.loc[k, 0.9]})
summary = pd.DataFrame(rows).set_index(["variable", "setting"]).unstack("setting").swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)
summary.round(3)

In [ ]:
LOGX = {"frequency", "effect_l2", "collateral_ctilde"}
fig, axes = plt.subplots(len(KEY), len(settings), figsize=(3.1 * len(settings), 2.0 * len(KEY)), sharey=False)
axes = np.atleast_2d(axes)
for j, s in enumerate(settings):
    for i, k in enumerate(KEY):
        ax = axes[i, j]
        v = feat[s][k].to_numpy()
        if k in LOGX:
            v = np.log10(v[v > 0] + 1e-12)
        ax.hist(v, bins=30, color=COLOR[s], edgecolor=SURFACE, linewidth=0.6)
        ax.set_yticks([])
        ax.grid(False)
        if i == 0:
            ax.set_title(PRETTY[s])
        if j == 0:
            ax.set_ylabel(LABEL[k] + (" (log10)" if k in LOGX else ""), fontsize=8.5, rotation=0, ha="right", va="center", labelpad=6)
fig.suptitle("Distributions of the main predictors and labels, 300 features per setting", x=0.01, ha="left", color=INK, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

In [ ]:
PRED = ["crowding", "crowd_max", "dec_norm", "enc_norm", "enc_dec_cos", "frequency", "act_mag", "act_std", "act_kurtosis",
        "coact_entropy", "coact_count", "logit_l2", "logit_linf", "logit_entropy"]
LAB = ["collateral_raw", "collateral_ctilde", "effect_l2", "stab_signed", "stab_abs", "kl_mean"]
cols = PRED + LAB
fig, axes = plt.subplots(1, len(settings), figsize=(5.4 * len(settings), 5.6))
axes = np.atleast_1d(axes)
for ax, s in zip(axes, settings):
    C = feat[s][cols].corr(method="spearman").to_numpy()
    im = ax.imshow(C, cmap=DIVERGING, vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90, fontsize=7)
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=7)
    ax.grid(False)
    ax.axhline(len(PRED) - 0.5, color=INK, lw=0.8); ax.axvline(len(PRED) - 0.5, color=INK, lw=0.8)
    for i in range(len(cols)):
        for j in range(len(cols)):
            if i != j and abs(C[i, j]) >= 0.5:
                ax.text(j, i, f"{C[i, j]:.1f}", ha="center", va="center", fontsize=5.5, color=INK if abs(C[i, j]) < 0.75 else SURFACE)
    ax.set_title(f"{PRETTY[s]}: Spearman correlations", loc="left")
cb = fig.colorbar(im, ax=axes.tolist(), fraction=0.012, pad=0.01)
cb.set_label("Spearman rho", color=INK2)
fig.suptitle("Predictors (top-left block) and steering labels (bottom-right block); the off-diagonal block is what the analysis is about", x=0.01, ha="left", color=INK, fontsize=12)
plt.show()

## 2. Crowding versus collateral

The paper's headline (Table 2) is that decoder crowding predicts the raw downstream collateral count in GPT-2-small (rho = 0.466). Section 3.4 names C-tilde, the count per unit of logit effect, as the primary collateral metric. Both are shown.

In [ ]:
fig, axes = plt.subplots(2, len(settings), figsize=(3.6 * len(settings), 6.6))
axes = np.atleast_2d(axes)
for j, s in enumerate(settings):
    df = feat[s]
    for i, (tgt, name, logy) in enumerate([("collateral_raw", "collateral count C", False), ("collateral_ctilde", "C-tilde = C / E_f", True)]):
        ax = axes[i, j]
        y = df[tgt].to_numpy()
        ax.scatter(df.crowding, y, s=22, color=COLOR[s], alpha=0.75, edgecolors=SURFACE, linewidths=0.8)
        if logy:
            ax.set_yscale("symlog", linthresh=max(np.percentile(y[y > 0], 5), 1e-3))
        r, p = stats.spearmanr(df.crowding, y)
        ax.text(0.02, 0.97, f"rho = {r:+.3f}", transform=ax.transAxes, va="top", color=INK, fontsize=10)
        if i == 0:
            ax.set_title(PRETTY[s])
        if j == 0:
            ax.set_ylabel(name)
        if i == 1:
            ax.set_xlabel("decoder crowding")
fig.suptitle("Decoder crowding against both collateral metrics (Spearman rho, n = 300 per panel)", x=0.01, ha="left", color=INK, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

In [ ]:
fig, axes = plt.subplots(2, len(settings), figsize=(3.6 * len(settings), 6.4))
axes = np.atleast_2d(axes)
for j, s in enumerate(settings):
    df = feat[s]
    for i, (x, y, xl, yl) in enumerate([("crowding", "effect_l2", "decoder crowding", "effect magnitude E_f"),
                                        ("effect_l2", "collateral_raw", "effect magnitude E_f", "collateral count C")]):
        ax = axes[i, j]
        ax.scatter(df[x], df[y], s=22, color=COLOR[s], alpha=0.75, edgecolors=SURFACE, linewidths=0.8)
        if x == "effect_l2":
            ax.set_xscale("log")
        if y == "effect_l2":
            ax.set_yscale("log")
        r, _ = stats.spearmanr(df[x], df[y])
        ax.text(0.02, 0.97, f"rho = {r:+.3f}", transform=ax.transAxes, va="top", color=INK, fontsize=10)
        if i == 0:
            ax.set_title(PRETTY[s])
        if j == 0:
            ax.set_ylabel(yl)
        ax.set_xlabel(xl)
fig.suptitle("Why effect magnitude is in the control set: crowded features steer harder, and harder steering moves more downstream features", x=0.01, ha="left", color=INK, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## 3. The control

Partial Spearman correlations: predictor and label are rank-transformed, both are residualized on the ranked control variables, and the residuals are correlated. Controls: none; the robust pair (frequency, activation magnitude); the primary set (the paper's Section 3.8 nuisance variables, effect magnitude E_f, intervention value and natural activation, plus firing frequency). Whiskers are 95% bootstrap intervals over the 300 features (10,000 resamples).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
controls = ["none", "robust", "primary"]
w = 0.24
for ax, (tgt, name) in zip(axes, [("collateral_raw", "collateral count C"), ("collateral_ctilde", "C-tilde = C / E_f")]):
    sub = part[(part.target == tgt) & (part.predictor == "crowding")]
    xs = np.arange(len(settings))
    for k, c in enumerate(controls):
        rows = [sub[(sub.setting == s) & (sub.control == c)].iloc[0] for s in settings]
        v = [r.rho for r in rows]
        ax.bar(xs + (k - 1) * w, v, w - 0.03, color=CONTROL_COLOR[c], label=CONTROL_LABEL[c], edgecolor=SURFACE, linewidth=2)
        ax.errorbar(xs + (k - 1) * w, v, yerr=[[r.rho - r.ci_lo for r in rows], [r.ci_hi - r.rho for r in rows]],
                    fmt="none", ecolor=INK2, elinewidth=1, capsize=3)
        if c == "primary":
            for x, r in zip(xs + w, rows):
                ax.annotate(f"{r.rho:+.2f}", (x, r.ci_hi), ha="center", va="bottom", fontsize=9, color=INK2, xytext=(0, 3), textcoords="offset points")
    ax.axhline(0, color=AXIS, lw=1)
    ax.set_xticks(xs); ax.set_xticklabels([PRETTY[s] for s in settings])
    ax.set_title(f"crowding vs {name}", loc="left")
axes[0].set_ylabel("Spearman rho (partial where controlled)")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper left", bbox_to_anchor=(0.01, 0.93), ncol=3, fontsize=9)
fig.suptitle("Decoder crowding survives the control in GPT-2-small, is null in Pythia, and loses most of its raw-count signal to effect magnitude in Gemma", x=0.01, ha="left", color=INK, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.88))
plt.show()

In [ ]:
fig, axes = plt.subplots(2, len(settings), figsize=(3.9 * len(settings), 6.8))
axes = np.atleast_2d(axes)
for j, s in enumerate(settings):
    for i, (tgt, name) in enumerate([("collateral_raw", "collateral count C"), ("collateral_ctilde", "C-tilde")]):
        ax = axes[i, j]
        sub = part[(part.setting == s) & (part.target == tgt) & (part.control == "primary")]
        sub = sub.iloc[(-sub.rho.abs()).argsort()[:6]].iloc[::-1]
        ys = np.arange(len(sub))
        ax.barh(ys, sub.rho, 0.6, color=COLOR[s], edgecolor=SURFACE, linewidth=2)
        ax.errorbar(sub.rho, ys, xerr=[sub.rho - sub.ci_lo, sub.ci_hi - sub.rho], fmt="none", ecolor=INK2, elinewidth=1, capsize=3)
        ax.set_yticks(ys); ax.set_yticklabels(sub.predictor, fontsize=8.5)
        ax.axvline(0, color=AXIS, lw=1)
        ax.set_xlim(-0.75, 0.75)
        if i == 0:
            ax.set_title(PRETTY[s])
        if j == 0:
            ax.set_ylabel(name)
        if i == 1:
            ax.set_xlabel("partial Spearman rho, primary control")
fig.suptitle("Strongest six predictors after the primary control; the leading predictor changes with the setting", x=0.01, ha="left", color=INK, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

## 4. Predictor sets on the residualized target

The paper's Table B3 residualizes the stability labels and re-runs its cross-validated ridge regressions on the residuals. This is the same procedure on the collateral labels: residualize on the primary control set, then predict with ridge (alpha = 1, standardized predictors) under five-fold cross-validation. Score = Spearman between held-out predictions and the residualized label, mean over folds, whiskers = fold standard deviation.

In [ ]:
SETS = ["frequency_only", "actmag_only", "coactivation_only", "direct_logit_only", "geometry_only", "full_no_magnitude", "full_all"]
fig, axes = plt.subplots(2, len(settings), figsize=(3.9 * len(settings), 6.4), sharex=True)
axes = np.atleast_2d(axes)
for j, s in enumerate(settings):
    for i, (tgt, name) in enumerate([("collateral_raw", "collateral count C"), ("collateral_ctilde", "C-tilde")]):
        ax = axes[i, j]
        sub = cvr[(cvr.setting == s) & (cvr.target == tgt) & (cvr.control == "primary")].set_index("predictor_set").loc[SETS]
        ys = np.arange(len(SETS))
        ax.barh(ys, sub.cv_spearman_mean, 0.6, color=COLOR[s], edgecolor=SURFACE, linewidth=2)
        ax.errorbar(sub.cv_spearman_mean, ys, xerr=sub.cv_spearman_sd, fmt="none", ecolor=INK2, elinewidth=1, capsize=3)
        ax.set_yticks(ys); ax.set_yticklabels([k.replace("_", " ") for k in SETS], fontsize=8.5)
        ax.axvline(0, color=AXIS, lw=1)
        if i == 0:
            ax.set_title(PRETTY[s])
        if j == 0:
            ax.set_ylabel(name)
        if i == 1:
            ax.set_xlabel("CV Spearman on the residualized label")
fig.suptitle("Table B3 analog: the no-magnitude set beats the frequency-only and activation-magnitude-only baselines in every setting", x=0.01, ha="left", color=INK, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

## 5. Robustness: feature samples, machines, protocol

Three checks. Different random samples of 300 features (seeds 0, 1, 2, same contexts). The same code on a Kaggle T4 with an older library stack. And the change from the original notebook's context construction (protocol v1, which left 33 of 2,048 GPT-2-small contexts ending in a pad token) to the reported protocol v2.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
# seeds
ax = axes[0]
sd = seeds[(seeds.target == "collateral_raw") & (seeds.n_seeds >= 2)]
xs = 0
ticks, labels = [], []
for s in [s for s in settings if s in set(sd.setting)]:
    for c in ["none", "robust", "primary"]:
        r = sd[(sd.setting == s) & (sd.control == c)]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        vals = [r[k] for k in ["0", "1", "2"] if k in r and pd.notna(r[k])]
        ax.scatter([xs] * len(vals), vals, s=40, color=CONTROL_COLOR[c], edgecolors=SURFACE, linewidths=1, zorder=3)
        ticks.append(xs); labels.append(f"{PRETTY[s]}\n{c}")
        xs += 1
    xs += 0.6
ax.axhline(0, color=AXIS, lw=1)
ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=7.5, rotation=90)
ax.set_ylabel("crowding vs collateral count, rho")
ax.set_title("Three feature samples per setting", loc="left")
# cross-machine
ax = axes[1]
if xcheck:
    keys = ["rho_crowding__collateral_raw", "partial_crowding__collateral_raw__given_freq_actmag", "rho_crowding__collateral_ctilde", "partial_crowding__collateral_ctilde__given_freq_actmag"]
    short = ["raw", "raw, partial", "C-tilde", "C-tilde, partial"]
    xs = np.arange(len(keys))
    for k, s in enumerate([s for s in settings if s in xcheck["headline"]]):
        a = [meta[s]["headline"][key] for key in keys]
        b = [xcheck["headline"][s][key] for key in keys]
        ax.scatter(xs + k * 0.25 - 0.12, a, s=44, color=COLOR[s], edgecolors=SURFACE, linewidths=1, label=f"{PRETTY[s]}, GB10", zorder=3)
        ax.scatter(xs + k * 0.25 - 0.12, b, s=44, facecolors=SURFACE, edgecolors=COLOR[s], linewidths=1.6, label=f"{PRETTY[s]}, Kaggle T4", zorder=3)
    ax.axhline(0, color=AXIS, lw=1)
    ax.set_xticks(xs); ax.set_xticklabels(short, fontsize=8.5)
    ax.legend(fontsize=8, loc="lower right", ncol=2)
    ax.set_title("Same code, two machines", loc="left")
# protocol
ax = axes[2]
if gate:
    keys = ["rho_crowding__collateral_raw", "partial_crowding__collateral_raw__given_freq_actmag", "rho_frequency__collateral_raw", "rho_act_mag__collateral_raw"]
    short = ["crowding", "crowding, partial", "frequency", "activation magnitude"]
    xs = np.arange(len(keys))
    v1 = [gate["headline"][k] for k in keys]
    v2 = [meta["gpt2_small"]["headline"][k] for k in keys]
    ax.bar(xs - 0.18, v1, 0.32, color="#86b6ef", edgecolor=SURFACE, linewidth=2, label="protocol v1 (original notebook)")
    ax.bar(xs + 0.18, v2, 0.32, color=COLOR["gpt2_small"], edgecolor=SURFACE, linewidth=2, label="protocol v2 (reported)")
    ax.axhline(0, color=AXIS, lw=1)
    ax.set_xticks(xs); ax.set_xticklabels(short, fontsize=8.5)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_ylim(0, 0.72)
    ax.set_title("GPT-2-small, context protocol", loc="left")
fig.suptitle("Robustness checks", x=0.01, ha="left", color=INK, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 6. Live reproduction on this GPU

The two ungated settings are recomputed here from scratch with the same script as the dataset (protocol v2, seed 0) and compared with the dataset's headline statistics. GPT-2-small takes a few minutes on a T4, Pythia-70M about one.

In [ ]:
import subprocess, sys, os, json, time
t0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformer-lens<3', 'sae-lens<6', 'transformers<5', 'pyyaml'], check=True)
import importlib.metadata as im
print('installed transformer-lens', im.version('transformer-lens'), '| sae-lens', im.version('sae-lens'), '| transformers', im.version('transformers'), '| torch', im.version('torch'), f'({time.time()-t0:.0f} s)')
os.makedirs('src', exist_ok=True); os.makedirs('configs', exist_ok=True); os.makedirs('results/logs', exist_ok=True)
open('src/run_setting.py', 'w').write(json.loads("\"#!/usr/bin/env python\\n\\\"\\\"\\\"Predictors and steering labels for one model/SAE setting of arXiv 2606.08365.\\n\\nComputes the intervention-free predictors for 300 sampled SAE features, steers each one\\nadditively at the final token (alpha = 1.0) and measures collateral, effect magnitude,\\nstability and KL shift. Writes results/<name>/per_feature.csv, selection.json, meta.json.\\n\\nProtocols: v1 rebuilds the contexts exactly as the original GPT-2-small notebook did (used\\nonly as a regression gate); v2 keeps texts with at least seq_len real tokens and deduplicates\\nthem, and is used for every reported setting.\\n\\\"\\\"\\\"\\nimport argparse\\nimport json\\nimport os\\nimport platform\\nimport re\\nimport sys\\nimport time\\n\\nimport numpy as np\\nimport torch\\nimport yaml\\n\\nT0 = time.time()\\nTIMINGS = {}\\n\\n\\ndef tick(msg, key=None):\\n    t = time.time() - T0\\n    print(f\\\"[{t:7.1f}s] {msg}\\\", flush=True)\\n    if key:\\n        TIMINGS[key] = round(t, 1)\\n\\n\\ndef _pkg_version(dist_name, module):\\n    try:\\n        from importlib.metadata import version\\n        return version(dist_name)\\n    except Exception:\\n        return getattr(module, \\\"__version__\\\", \\\"unknown\\\")\\n\\n\\ndef sae_field(sae, name, default=None):\\n    \\\"\\\"\\\"Read a config field across SAELens versions (cfg.metadata.<name>, cfg.<name>, dict).\\\"\\\"\\\"\\n    cfg = sae.cfg\\n    for obj in (getattr(cfg, \\\"metadata\\\", None), cfg):\\n        if obj is not None and hasattr(obj, name):\\n            v = getattr(obj, name)\\n            if v is not None:\\n                return v\\n    try:\\n        d = cfg.to_dict()\\n        if d.get(name) is not None:\\n            return d[name]\\n        md = d.get(\\\"metadata\\\")\\n        if isinstance(md, dict) and md.get(name) is not None:\\n            return md[name]\\n    except Exception:\\n        pass\\n    return default\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--config\\\", required=True)\\n    ap.add_argument(\\\"--out\\\", default=None, help=\\\"output dir (default results/<name>)\\\")\\n    ap.add_argument(\\\"--smoke\\\", action=\\\"store_true\\\", help=\\\"tiny sizes to validate the pipeline\\\")\\n    ap.add_argument(\\\"--protocol\\\", choices=[\\\"v1\\\", \\\"v2\\\"], default=\\\"v2\\\")\\n    ap.add_argument(\\\"--dtype\\\", default=None, help=\\\"override model dtype: float32|bfloat16|float16\\\")\\n    ap.add_argument(\\\"--device\\\", default=\\\"cuda\\\")\\n    ap.add_argument(\\\"--seed\\\", type=int, default=0)\\n    args = ap.parse_args()\\n\\n    cfg = yaml.safe_load(open(args.config))\\n    name = cfg[\\\"name\\\"]\\n    out_dir = args.out or os.path.join(\\\"results\\\", name + (\\\"_smoke\\\" if args.smoke else \\\"\\\") +\\n                                       (\\\"_v1\\\" if args.protocol == \\\"v1\\\" else \\\"\\\"))\\n    os.makedirs(out_dir, exist_ok=True)\\n\\n    if args.smoke:\\n        N_TEXTS, N_CONTEXTS, N_FEATURES, CTX_PER_TYPE, PANEL = 600, 128, 12, 4, 256\\n    else:\\n        N_TEXTS, N_CONTEXTS, N_FEATURES = cfg[\\\"n_texts\\\"], cfg[\\\"n_contexts\\\"], cfg[\\\"n_features\\\"]\\n        CTX_PER_TYPE, PANEL = cfg[\\\"ctx_per_type\\\"], cfg[\\\"panel_size\\\"]\\n    SEQ_LEN = cfg[\\\"seq_len\\\"]\\n    ALPHA = float(cfg[\\\"alpha\\\"])\\n    TAU = float(cfg[\\\"tau\\\"])\\n    EPS_FIRE = float(cfg[\\\"eps_fire\\\"])\\n    TOPK_CROWD = int(cfg[\\\"topk_crowding\\\"])\\n    FREQ_LO, FREQ_HI = cfg[\\\"freq_band\\\"]\\n    MIN_CHARS = int(cfg.get(\\\"min_text_chars\\\", 200))\\n    MODEL_BATCH = int(cfg.get(\\\"model_batch\\\", 32))\\n    SEED = args.seed\\n    dtype_name = args.dtype or cfg.get(\\\"dtype\\\", \\\"float32\\\")\\n    dtype = getattr(torch, dtype_name)\\n    device = args.device\\n\\n    print(\\\"config:\\\", json.dumps(dict(name=name, protocol=args.protocol, smoke=args.smoke,\\n                                     N_TEXTS=N_TEXTS, N_CONTEXTS=N_CONTEXTS, N_FEATURES=N_FEATURES,\\n                                     CTX_PER_TYPE=CTX_PER_TYPE, PANEL=PANEL, SEQ_LEN=SEQ_LEN,\\n                                     ALPHA=ALPHA, TAU=TAU, TOPK_CROWD=TOPK_CROWD,\\n                                     FREQ_BAND=[FREQ_LO, FREQ_HI], dtype=dtype_name, seed=SEED)))\\n\\n    import transformer_lens\\n    import sae_lens\\n    import transformers\\n    from transformer_lens import HookedTransformer\\n    from sae_lens import SAE\\n    from datasets import load_dataset\\n    import scipy.stats as st\\n\\n    torch.set_grad_enabled(False)\\n    np.random.seed(SEED)\\n    torch.manual_seed(SEED)\\n    assert torch.cuda.is_available() or device == \\\"cpu\\\", \\\"no CUDA device found\\\"\\n    tick(f\\\"device {torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu'}; loading SAEs\\\")\\n\\n    # ---- SAEs first: the model is loaded with the kwargs the SAE release expects ----\\n    def load_sae(sae_id):\\n        out = SAE.from_pretrained(release=cfg[\\\"sae_release\\\"], sae_id=sae_id, device=device)\\n        sae = out[0] if isinstance(out, (tuple, list)) else out\\n        return sae.eval()\\n\\n    sae_p = load_sae(cfg[\\\"primary_sae_id\\\"])\\n    sae_d = load_sae(cfg[\\\"downstream_sae_id\\\"])\\n    hook_p = sae_field(sae_p, \\\"hook_name\\\")\\n    hook_d = sae_field(sae_d, \\\"hook_name\\\")\\n    assert hook_p == cfg[\\\"primary_hook\\\"], f\\\"primary SAE hook {hook_p} != config {cfg['primary_hook']}\\\"\\n    assert hook_d == cfg[\\\"downstream_hook\\\"], f\\\"downstream SAE hook {hook_d} != config {cfg['downstream_hook']}\\\"\\n    prepend_bos = bool(sae_field(sae_p, \\\"prepend_bos\\\", True))\\n    model_kwargs = dict(sae_field(sae_p, \\\"model_from_pretrained_kwargs\\\", {}) or {})\\n    sae_meta = {\\n        \\\"primary\\\": {\\\"hook_name\\\": hook_p, \\\"d_sae\\\": int(sae_p.cfg.d_sae), \\\"d_in\\\": int(sae_p.cfg.d_in),\\n                    \\\"architecture\\\": str(sae_field(sae_p, \\\"architecture\\\", type(sae_p).__name__)),\\n                    \\\"prepend_bos\\\": prepend_bos, \\\"model_from_pretrained_kwargs\\\": model_kwargs},\\n        \\\"downstream\\\": {\\\"hook_name\\\": hook_d, \\\"d_sae\\\": int(sae_d.cfg.d_sae), \\\"d_in\\\": int(sae_d.cfg.d_in),\\n                       \\\"architecture\\\": str(sae_field(sae_d, \\\"architecture\\\", type(sae_d).__name__))},\\n    }\\n    tick(f\\\"SAEs loaded: primary d_sae={sae_p.cfg.d_sae} ({hook_p}), downstream d_sae={sae_d.cfg.d_sae} ({hook_d}); \\\"\\n         f\\\"model_from_pretrained_kwargs={model_kwargs}; prepend_bos={prepend_bos}\\\", \\\"saes_loaded\\\")\\n\\n    # transformers 5 renamed NeoX's embed_out to lm_head; TransformerLens still reads embed_out.\\n    hf_model = None\\n    if \\\"pythia\\\" in cfg[\\\"model_name\\\"].lower():\\n        from transformers import AutoModelForCausalLM\\n        hf_name = cfg.get(\\\"hf_model_name\\\", \\\"EleutherAI/\\\" + cfg[\\\"model_name\\\"])\\n        hf_model = AutoModelForCausalLM.from_pretrained(hf_name, torch_dtype=dtype)\\n        if not hasattr(hf_model, \\\"embed_out\\\") and hasattr(hf_model, \\\"lm_head\\\"):\\n            hf_model.embed_out = hf_model.lm_head\\n    elif \\\"llama\\\" in cfg[\\\"model_name\\\"].lower():\\n        from transformers import AutoModelForCausalLM\\n        hf_model = AutoModelForCausalLM.from_pretrained(cfg[\\\"model_name\\\"], torch_dtype=dtype, low_cpu_mem_usage=True)\\n    # tl_processing: \\\"default\\\" applies TransformerLens weight processing; \\\"none\\\" skips it, which\\n    # roughly halves peak memory for large models and leaves the residual stream unchanged.\\n    tl_processing = cfg.get(\\\"tl_processing\\\", \\\"default\\\")\\n    loader = HookedTransformer.from_pretrained_no_processing if tl_processing == \\\"none\\\" else HookedTransformer.from_pretrained\\n    model = loader(cfg[\\\"model_name\\\"], hf_model=hf_model, device=device, dtype=dtype, **model_kwargs).eval()\\n    del hf_model\\n    import gc\\n    gc.collect()\\n    if device == \\\"cuda\\\":\\n        torch.cuda.empty_cache()\\n    tick(f\\\"model loaded: {cfg['model_name']} d_model={model.cfg.d_model} n_layers={model.cfg.n_layers} \\\"\\n         f\\\"norm={model.cfg.normalization_type} dtype={dtype_name}\\\", \\\"model_loaded\\\")\\n    W_dec = sae_p.W_dec.detach().float()                    # [d_sae, d_model]\\n    W_enc = sae_p.W_enc.detach().float()                    # [d_model, d_sae]\\n    d_sae, d_model = W_dec.shape\\n    assert d_model == model.cfg.d_model\\n\\n    # ---- contexts ----\\n    tick(\\\"streaming wikitext-103 train split\\\")\\n    ds = load_dataset(\\\"Salesforce/wikitext\\\", \\\"wikitext-103-raw-v1\\\", split=\\\"train\\\", streaming=True)\\n    texts = []\\n    seen = set()\\n    n_dupes = 0\\n    for row in ds:\\n        t = row[\\\"text\\\"].strip()\\n        if len(t) <= MIN_CHARS:\\n            continue\\n        if args.protocol == \\\"v2\\\":\\n            key = re.sub(r\\\"\\\\s+\\\", \\\" \\\", t)\\n            if key in seen:\\n                n_dupes += 1\\n                continue\\n            seen.add(key)\\n        texts.append(t)\\n        if len(texts) >= N_TEXTS:\\n            break\\n    tick(f\\\"collected {len(texts)} texts (dupes removed: {n_dupes}); tokenising and caching clean activations\\\")\\n\\n    def encode_final(sae, resid):                            # [B, d_model] -> [B, d_sae]\\n        return sae.encode(resid.to(device).float())\\n\\n    tok_rows = []\\n    n_short_dropped = 0\\n    pad_id = model.tokenizer.pad_token_id\\n    if args.protocol == \\\"v1\\\":\\n        batches = []\\n        made = 0\\n        for i in range(0, len(texts), 32):\\n            toks = model.to_tokens(texts[i:i + 32], prepend_bos=prepend_bos)\\n            if toks.shape[1] < SEQ_LEN:\\n                continue\\n            batches.append(toks[:, :SEQ_LEN].cpu())\\n            made += toks.shape[0]\\n            if made >= N_CONTEXTS:\\n                break\\n        all_toks = torch.cat(batches)[:N_CONTEXTS]\\n    else:\\n        for t in texts:\\n            toks = model.to_tokens(t, prepend_bos=prepend_bos)[0]\\n            if toks.shape[0] < SEQ_LEN:\\n                n_short_dropped += 1\\n                continue\\n            tok_rows.append(toks[:SEQ_LEN].cpu())\\n            if len(tok_rows) >= N_CONTEXTS:\\n                break\\n        all_toks = torch.stack(tok_rows)\\n    N = all_toks.shape[0]\\n    assert N == N_CONTEXTS, f\\\"only {N} contexts built, wanted {N_CONTEXTS}: raise n_texts\\\"\\n    n_final_pad = int((all_toks[:, -1] == pad_id).sum().item()) if pad_id is not None else -1\\n    tick(f\\\"{N} contexts x {SEQ_LEN} tokens; short texts dropped (v2): {n_short_dropped}; \\\"\\n         f\\\"contexts whose final token is the pad token: {n_final_pad}\\\")\\n\\n    prim_clean, down_clean = [], []\\n    for i in range(0, N, MODEL_BATCH):\\n        toks = all_toks[i:i + MODEL_BATCH].to(device)\\n        _, cache = model.run_with_cache(toks, names_filter=[hook_p, hook_d])\\n        prim_clean.append(encode_final(sae_p, cache[hook_p][:, -1, :]).cpu())\\n        down_clean.append(encode_final(sae_d, cache[hook_d][:, -1, :]).cpu())\\n        del cache\\n    prim_clean = torch.cat(prim_clean)                       # [N, d_sae]  primary-site feature acts\\n    down_clean = torch.cat(down_clean)                       # [N, d_sae_down]\\n    l0_p = (prim_clean > EPS_FIRE).float().sum(1).mean().item()\\n    l0_d = (down_clean > EPS_FIRE).float().sum(1).mean().item()\\n    dead_p = ((prim_clean > EPS_FIRE).float().sum(0) == 0).float().mean().item()\\n    tick(f\\\"cached clean final-token activations: prim {tuple(prim_clean.shape)} down {tuple(down_clean.shape)}; \\\"\\n         f\\\"mean L0 primary={l0_p:.1f} downstream={l0_d:.1f}; primary features never firing={dead_p:.1%}\\\", \\\"cache_done\\\")\\n\\n    # ---- Phase 1: intervention-free predictors ----\\n    freq = (prim_clean > EPS_FIRE).float().mean(0).numpy()          # final-token firing frequency\\n    act_mag = prim_clean.mean(0).numpy()                            # mean final-token activation\\n\\n    Wn = torch.nn.functional.normalize(W_dec, dim=-1)\\n    crowd = np.empty(d_sae, dtype=np.float32)\\n    crowd_max = np.empty(d_sae, dtype=np.float32)\\n    for chunk in torch.split(torch.arange(d_sae), 2048):\\n        sims = (Wn[chunk] @ Wn.T).abs()\\n        sims[torch.arange(len(chunk)), chunk] = 0.0                  # exclude self\\n        top = torch.topk(sims, TOPK_CROWD, dim=-1).values\\n        crowd[chunk.numpy()] = top.mean(-1).cpu().numpy()\\n        crowd_max[chunk.numpy()] = top[:, 0].cpu().numpy()\\n        del sims\\n    tick(\\\"predictors: crowding done\\\")\\n\\n    rng = np.random.default_rng(SEED)                                 # same call order as the notebook\\n    elig = np.where((freq >= FREQ_LO) & (freq <= FREQ_HI))[0]\\n    feats = rng.choice(elig, size=min(N_FEATURES, len(elig)), replace=False)\\n    feats.sort()\\n    print(f\\\"eligible features: {len(elig)} of {d_sae} | sampled: {len(feats)}\\\")\\n\\n    down_freq = (down_clean > EPS_FIRE).float().mean(0)\\n    panel = torch.topk(down_freq, min(PANEL, down_clean.shape[1])).indices.to(device)\\n\\n    F = torch.as_tensor(feats)\\n    A = prim_clean[:, F].float()                                      # [N, 300] sampled feature acts\\n    fires = (A > EPS_FIRE).float()\\n    n_fire = fires.sum(0).clamp(min=1)\\n    act_mean_firing = ((A * fires).sum(0) / n_fire).numpy()\\n    act_std = A.std(0).numpy()\\n    act_max = A.max(0).values.numpy()\\n    act_kurt = st.kurtosis(A.numpy(), axis=0, fisher=True, bias=True)\\n    p_f = freq[feats]\\n    eps = 1e-12\\n    bin_entropy = -(p_f * np.log(p_f + eps) + (1 - p_f) * np.log(1 - p_f + eps))\\n    r = A / (A.sum(0, keepdim=True) + eps)\\n    act_entropy = (-(r * torch.log(r + eps)).sum(0) / np.log(N)).numpy()\\n\\n    B = (prim_clean > EPS_FIRE).float().to(device)                    # [N, d_sae]\\n    Bf = B[:, F.to(device)]                                            # [N, 300]\\n    co = (Bf.T @ B) / (Bf.sum(0, keepdim=True).T + eps)                # q_{f,j}  [300, d_sae]\\n    co[torch.arange(len(feats)), F.to(device)] = 0.0\\n    pi = co / (co.sum(1, keepdim=True) + eps)\\n    coact_entropy = (-(pi * torch.log(pi + eps)).sum(1)).cpu().numpy()\\n    coact_count = ((Bf.T @ B.sum(1, keepdim=True)).squeeze(1) / (Bf.sum(0) + eps) - 1).cpu().numpy()\\n    del B, Bf, co, pi\\n\\n    W_U = model.W_U.detach().float()                                  # [d_model, vocab]\\n    r_f = W_dec[F.to(device)] @ W_U                                    # direct-logit vectors [300, vocab]\\n    logit_l2 = r_f.norm(dim=-1).cpu().numpy()\\n    logit_linf = r_f.abs().max(-1).values.cpu().numpy()\\n    s = r_f.abs() / (r_f.abs().sum(-1, keepdim=True) + eps)\\n    logit_entropy = (-(s * torch.log(s + eps)).sum(-1)).cpu().numpy()\\n    logit_top10_mass = torch.topk(s, 10, dim=-1).values.sum(-1).cpu().numpy()\\n    del r_f, s\\n\\n    dec_norm = W_dec[F.to(device)].norm(dim=-1).cpu().numpy()\\n    enc_vec = W_enc[:, F.to(device)].T                                 # [300, d_model]\\n    enc_norm = enc_vec.norm(dim=-1).cpu().numpy()\\n    enc_dec_cos = torch.nn.functional.cosine_similarity(enc_vec, W_dec[F.to(device)], dim=-1).cpu().numpy()\\n    tick(\\\"predictors: all done\\\", \\\"predictors_done\\\")\\n\\n    # ---- Phase 2: steering labels on the mixed context set ----\\n    def pick_contexts(fi):\\n        a = prim_clean[:, fi].numpy()\\n        order = np.argsort(-a)\\n        top = order[:CTX_PER_TYPE]\\n        low = order[-CTX_PER_TYPE:]\\n        mid = rng.choice(np.setdiff1d(order, np.concatenate([top, low])), size=CTX_PER_TYPE, replace=False)\\n        return np.concatenate([top, mid, low])\\n\\n    def steered_forward(toks, d_f):\\n        store = {}\\n\\n        def steer(resid, hook):\\n            resid[:, -1, :] = resid[:, -1, :] + ALPHA * d_f.to(resid.dtype)\\n            return resid\\n\\n        def grab(resid, hook):\\n            store[\\\"d\\\"] = resid[:, -1, :].detach()\\n            return resid\\n\\n        logits = model.run_with_hooks(toks, return_type=\\\"logits\\\", fwd_hooks=[(hook_p, steer), (hook_d, grab)])\\n        return logits[:, -1, :].detach().float(), store[\\\"d\\\"]\\n\\n    rows = []\\n    n_ctx = 3 * CTX_PER_TYPE\\n    tick(f\\\"steering {len(feats)} features x {n_ctx} contexts\\\")\\n    for n, fi in enumerate(feats):\\n        ctx = pick_contexts(fi)\\n        toks = all_toks[ctx].to(device)\\n        d_f = W_dec[fi]\\n        logit_c, cache = model.run_with_cache(toks, names_filter=[hook_d], return_type=\\\"logits\\\")\\n        logit_c = logit_c[:, -1, :].float()\\n        u_clean = encode_final(sae_d, cache[hook_d][:, -1, :])\\n        del cache\\n        logit_s, down_resid = steered_forward(toks, d_f)\\n        u_steer = encode_final(sae_d, down_resid)\\n        du = (u_steer - u_clean)[:, panel].abs()                      # [n_ctx, PANEL]\\n        coll = (du > TAU).float().sum(-1).mean().item()               # C_{f,tau}\\n        dl = logit_s - logit_c                                         # [n_ctx, vocab]\\n        dl_norm = dl.norm(dim=-1)\\n        Ef = dl_norm.mean().item()                                     # E_f\\n        dl_mean = dl.mean(0, keepdim=True)\\n        cos = torch.nn.functional.cosine_similarity(dl, dl_mean.expand_as(dl), dim=-1)\\n        logp = torch.log_softmax(logit_c, -1)\\n        logq = torch.log_softmax(logit_s, -1)\\n        kl = (logp.exp() * (logp - logq)).sum(-1).mean().item()\\n        rows.append({\\n            \\\"feature\\\": int(fi),\\n            \\\"crowding\\\": float(crowd[fi]), \\\"crowd_max\\\": float(crowd_max[fi]),\\n            \\\"dec_norm\\\": float(dec_norm[n]), \\\"enc_norm\\\": float(enc_norm[n]), \\\"enc_dec_cos\\\": float(enc_dec_cos[n]),\\n            \\\"frequency\\\": float(freq[fi]), \\\"act_mag\\\": float(act_mag[fi]),\\n            \\\"act_mean_firing\\\": float(act_mean_firing[n]), \\\"act_std\\\": float(act_std[n]), \\\"act_max\\\": float(act_max[n]),\\n            \\\"act_kurtosis\\\": float(act_kurt[n]), \\\"bin_entropy\\\": float(bin_entropy[n]), \\\"act_entropy\\\": float(act_entropy[n]),\\n            \\\"coact_entropy\\\": float(coact_entropy[n]), \\\"coact_count\\\": float(coact_count[n]),\\n            \\\"logit_l2\\\": float(logit_l2[n]), \\\"logit_linf\\\": float(logit_linf[n]),\\n            \\\"logit_entropy\\\": float(logit_entropy[n]), \\\"logit_top10_mass\\\": float(logit_top10_mass[n]),\\n            \\\"collateral_raw\\\": coll, \\\"effect_l2\\\": Ef, \\\"collateral_ctilde\\\": coll / (Ef + 1e-8),\\n            \\\"stab_signed\\\": cos.mean().item(), \\\"stab_abs\\\": cos.abs().mean().item(),\\n            \\\"kl_mean\\\": kl, \\\"kl_per_effect\\\": kl / (Ef + 1e-8),\\n            \\\"effect_cv\\\": (dl_norm.std() / (dl_norm.mean() + 1e-8)).item(),\\n            \\\"ctx_mean_act\\\": float(prim_clean[ctx, fi].mean().item()),\\n            \\\"intervention_value\\\": ALPHA, \\\"n_ctx\\\": int(n_ctx),\\n        })\\n        if (n + 1) % max(1, len(feats) // 6) == 0:\\n            tick(f\\\"  steered {n + 1}/{len(feats)}\\\")\\n    tick(\\\"steering done\\\", \\\"steering_done\\\")\\n\\n    import pandas as pd\\n    df = pd.DataFrame(rows)\\n    df.to_csv(os.path.join(out_dir, \\\"per_feature.csv\\\"), index=False)\\n    json.dump({\\\"features\\\": [int(x) for x in feats], \\\"panel\\\": [int(x) for x in panel.cpu().numpy()]},\\n              open(os.path.join(out_dir, \\\"selection.json\\\"), \\\"w\\\"))\\n\\n    # ---- headline statistics (regression gate) ----\\n    from scipy.stats import rankdata, pearsonr\\n\\n    def sp(x, y):\\n        r_, p_ = st.spearmanr(df[x], df[y])\\n        return round(float(r_), 4), float(p_)\\n\\n    def partial(x, y, zs):\\n        def resid(a, Z):\\n            Z1 = np.c_[np.ones(len(a)), Z]\\n            coef, *_ = np.linalg.lstsq(Z1, a, rcond=None)\\n            return a - Z1 @ coef\\n        Z = np.c_[[rankdata(df[z]) for z in zs]].T\\n        r_, p_ = pearsonr(resid(rankdata(df[x]), Z), resid(rankdata(df[y]), Z))\\n        return round(float(r_), 4), float(p_)\\n\\n    head = {}\\n    for tgt in [\\\"collateral_raw\\\", \\\"collateral_ctilde\\\"]:\\n        for pred in [\\\"crowding\\\", \\\"frequency\\\", \\\"act_mag\\\"]:\\n            head[f\\\"rho_{pred}__{tgt}\\\"] = sp(pred, tgt)[0]\\n        head[f\\\"partial_crowding__{tgt}__given_freq_actmag\\\"] = partial(\\\"crowding\\\", tgt, [\\\"frequency\\\", \\\"act_mag\\\"])[0]\\n    med = df[\\\"frequency\\\"].median()\\n    lo, hi = df[df.frequency <= med], df[df.frequency > med]\\n    head[\\\"crowd_rho_lowfreq_half\\\"] = round(float(st.spearmanr(lo.crowding, lo.collateral_raw)[0]), 4)\\n    head[\\\"crowd_rho_highfreq_half\\\"] = round(float(st.spearmanr(hi.crowding, hi.collateral_raw)[0]), 4)\\n    print(json.dumps(head, indent=1))\\n\\n    meta = {\\n        \\\"setting\\\": name, \\\"protocol\\\": args.protocol, \\\"smoke\\\": args.smoke, \\\"seed\\\": SEED, \\\"config\\\": cfg,\\n        \\\"sizes\\\": dict(n_texts=len(texts), n_contexts=N, seq_len=SEQ_LEN, n_features=len(feats), n_eligible=int(len(elig)),\\n                      ctx_per_type=CTX_PER_TYPE, n_ctx_per_feature=n_ctx, panel=int(panel.numel()),\\n                      d_sae_primary=int(d_sae), d_sae_downstream=int(down_clean.shape[1]),\\n                      mean_l0_primary=round(l0_p, 2), mean_l0_downstream=round(l0_d, 2), frac_primary_never_firing=round(dead_p, 4)),\\n        \\\"context_build\\\": dict(dupes_removed=n_dupes, short_texts_dropped=n_short_dropped,\\n                              contexts_final_token_is_pad=n_final_pad, pad_token_id=pad_id),\\n        \\\"sae\\\": sae_meta, \\\"dtype\\\": dtype_name, \\\"tl_processing\\\": tl_processing, \\\"device\\\": torch.cuda.get_device_name(0) if device == \\\"cuda\\\" else \\\"cpu\\\",\\n        \\\"versions\\\": dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,\\n                         transformer_lens=_pkg_version(\\\"transformer-lens\\\", transformer_lens),\\n                         sae_lens=_pkg_version(\\\"sae-lens\\\", sae_lens), numpy=np.__version__),\\n        \\\"timings_s\\\": TIMINGS, \\\"wall_clock_s\\\": round(time.time() - T0, 1), \\\"headline\\\": head,\\n    }\\n    json.dump(meta, open(os.path.join(out_dir, \\\"meta.json\\\"), \\\"w\\\"), indent=1)\\n    tick(f\\\"wrote {out_dir}/per_feature.csv, selection.json, meta.json\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\""))
open('configs/pythia_70m_deduped.yaml', 'w').write(json.loads("\"# Pythia-70M-deduped, paper Appendix Table A1 / A2 (arXiv 2606.08365)\\nname: pythia_70m_deduped\\nmodel_name: pythia-70m-deduped\\nsae_release: pythia-70m-deduped-res-sm\\nprimary_sae_id: blocks.4.hook_resid_post\\ndownstream_sae_id: blocks.5.hook_resid_post\\nprimary_hook: blocks.4.hook_resid_post\\ndownstream_hook: blocks.5.hook_resid_post\\npanel_size: 2048            # Evan Duan, email 2026-08-26: DOWNSTREAM_PANEL_SIZE = 2048\\ndtype: float32\\nmodel_batch: 32\\nn_texts: 8000               # Table A2: 8,000 distinct texts\\nn_contexts: 2048            # Table A2: 2,048 contexts\\nseq_len: 48                 # Table A2: context length 48\\nn_features: 300             # Table A2: 300 features\\nctx_per_type: 16            # Table A2: 16 contexts per type (top / random / low)\\nalpha: 1.0                  # Section 3.3: fixed_global_add, alpha = 1.0\\ntau: 0.05                   # Section 3.4: collateral threshold\\neps_fire: 1.0e-6            # Section 3.2\\ntopk_crowding: 20           # Evan Duan, email 2026-08-26: TOPK_CROWDING = 20\\nfreq_band: [0.002, 0.50]    # ours: the \\\"non-degenerate range\\\" of Section 3.2 is not quantified in the paper\\nmin_text_chars: 200\\n\""))
open('configs/gpt2_small.yaml', 'w').write(json.loads("\"# GPT-2-small, paper Appendix Table A1 / A2 (arXiv 2606.08365)\\nname: gpt2_small\\nmodel_name: gpt2\\nsae_release: gpt2-small-res-jb\\nprimary_sae_id: blocks.8.hook_resid_pre\\ndownstream_sae_id: blocks.10.hook_resid_pre\\nprimary_hook: blocks.8.hook_resid_pre\\ndownstream_hook: blocks.10.hook_resid_pre\\npanel_size: 2048            # Evan Duan, email 2026-08-26: DOWNSTREAM_PANEL_SIZE = 2048\\ndtype: float32\\nmodel_batch: 32\\nn_texts: 8000               # Table A2: 8,000 distinct texts\\nn_contexts: 2048            # Table A2: 2,048 contexts\\nseq_len: 48                 # Table A2: context length 48\\nn_features: 300             # Table A2: 300 features\\nctx_per_type: 16            # Table A2: 16 contexts per type (top / random / low)\\nalpha: 1.0                  # Section 3.3: fixed_global_add, alpha = 1.0\\ntau: 0.05                   # Section 3.4: collateral threshold\\neps_fire: 1.0e-6            # Section 3.2\\ntopk_crowding: 20           # Evan Duan, email 2026-08-26: TOPK_CROWDING = 20\\nfreq_band: [0.002, 0.50]    # ours: the \\\"non-degenerate range\\\" of Section 3.2 is not quantified in the paper\\nmin_text_chars: 200\\n\""))
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

In [ ]:
!python src/run_setting.py --config configs/pythia_70m_deduped.yaml --protocol v2 --out results/live_pythia 2>&1 | grep -v 'Warning\|warn\|Loading weights' | tail -18

In [ ]:
!python src/run_setting.py --config configs/gpt2_small.yaml --protocol v2 --out results/live_gpt2 2>&1 | grep -v 'Warning\|warn\|Loading weights' | tail -18

In [ ]:
rows = []
for s, live in [("pythia_70m_deduped", "results/live_pythia/meta.json"), ("gpt2_small", "results/live_gpt2/meta.json")]:
    if not os.path.exists(live):
        continue
    L = json.load(open(live))
    for k, v in meta[s]["headline"].items():
        rows.append({"setting": PRETTY[s], "statistic": k, "dataset (GB10)": v, "this Kaggle run": L["headline"][k], "difference": L["headline"][k] - v})
    print(PRETTY[s], "| eligible features here:", L["sizes"]["n_eligible"], "vs dataset:", meta[s]["sizes"]["n_eligible"],
          "| mean L0 here:", L["sizes"].get("mean_l0_primary"), "vs dataset:", meta[s]["sizes"].get("mean_l0_primary"), "| device:", L["device"])
pd.DataFrame(rows).round(3)

## Notes

Natural activation is taken as the mean final-token activation over the 2,048 contexts, and the intervention value is constant under fixed_global_add with alpha = 1.0, so it drops out of the regression. Feature sampling uses seed 0 over the firing-frequency band [0.002, 0.50]. The downstream panel is the most frequently active downstream features on clean contexts (2,048; 1,024 for Llama). Gemma-2-2B and Llama-3.1-8B are loaded from Hugging Face under their gated licences on the original machine; Llama runs in bfloat16 without TransformerLens weight processing to fit in memory, which leaves the residual stream the SAEs read unchanged. Full write-up and LaTeX table: `RESULTS.md` in the repository.